# Mojito Gaps Investigation Pipeline

Demonstrates the full TDI data processing pipeline for gapped data using
`MojitoProcessor` and `MojitoProcessor.gaps`.

The pipeline applies the following steps in order:

| Step | Operation | Notes |
|------|-----------|-------|
| 1 | Generate gap mask | `lisaglitch` + `lisagap` |
| 2 | Apply mask to raw data | `gaps.apply_raw_mask` |
| 3 | Run processing pipeline | filter → downsample → trim → segment → window |
| 4 | Compute extended mask | `gaps.compute_extended_mask` (filter leakage) |
| 5 | Taper mask | `gaps.taper_mask` |
| 6 | Apply final mask | `gaps.apply_mask_to_processor` |
| 7 | Derive AET & compute PSD | `sp.to_aet()`, `sp.fft()` |

The `gap_contamination` array from step 4 is kept as a separate variable so
it can be inspected and the threshold tuned before committing.

In [ ]:
import logging

import numpy as np
import matplotlib.pyplot as plt

from lisaglitch import GapMaskGenerator
from lisagap import GapWindowGenerator
from scipy.signal import windows

import MojitoProcessor as mp
import MojitoProcessor.gaps as gaps

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")

## 1. Load Data

In [ ]:
mojito_data_file = (
    "../Mojito_Data/NOISE_731d_0.25s_L1_source0_0_20251206T220508924302Z.h5"
)

# Set to a float (days) to load only a subset of the file
load_days = None

data = mp.io.load_file(mojito_data_file, load_days=load_days)

n_samples = len(data["tdis"]["X"])
duration = n_samples * data["dt"]
print(f"Loaded: {n_samples:,} samples @ {data['fs']} Hz ({duration / 86400:.2f} days)")
print(f"TDI channels: {list(data['tdis'].keys())}")

## 2. Generate Gap Mask

Use `lisaglitch.GapMaskGenerator` to define planned gaps (antenna repointing),
then wrap with `lisagap.GapWindowGenerator` to produce a smooth tapering mask.
A Tukey window is applied over the whole dataset to bring the endpoints to zero.

In [ ]:
# ── Gap schedule ──────────────────────────────────────────────────────────────
gap_definitions = {
    "planned": {
        "antenna_repointing": {
            "rate_per_year": 26,
            "duration_hr": 7,
        }
    },
    "unplanned": {},
}

taper_definitions = {
    "planned": {
        "antenna_repointing": {"lobe_lengths_hr": 3.0},
    },
    "unplanned": {},
}

# ── Build smoothed mask at the raw sampling rate ──────────────────────────────
gap_gen = GapMaskGenerator(
    sim_t=data["t_tdi"],
    gap_definitions=gap_definitions,
    treat_as_nan=False,
)
window_func = GapWindowGenerator(gap_gen)
smoothed_mask = window_func.generate_window(
    include_planned=True,
    include_unplanned=False,
    apply_tapering=True,
    taper_definitions=taper_definitions,
)[0]

# Multiply by a dataset-wide Tukey window to bring the endpoints to zero
smoothed_mask = windows.tukey(len(smoothed_mask), alpha=0.01) * smoothed_mask

print(f"Gap fraction in raw mask: {1 - smoothed_mask.mean():.4%}")

## 3. Apply Mask to Raw Data

`gaps.apply_raw_mask` deep-copies the data dict and zeros out every TDI channel
where the mask is zero. The original `data` dict is unchanged.

In [ ]:
data_gapped = gaps.apply_raw_mask(data, smoothed_mask)

plt.figure(figsize=(14, 3))
plt.plot(data["t_tdi"][::50] / 3600, data_gapped["tdis"]["X"][::50], label="X (gapped)")
plt.plot(
    data["t_tdi"][::50] / 3600,
    np.max(np.abs(data["tdis"]["X"])) * smoothed_mask[::50],
    label="Smoothed mask",
    alpha=0.7,
)
plt.xlabel("Time (hours)")
plt.ylabel("Signal")
plt.title("TDI-X with gap mask applied")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Run the Processing Pipeline

All pipeline parameters are configurable here. The pipeline runs in this order:

```
bandpass/highpass filter → downsample → trim → truncate → window
```

The time-domain data is normalised by the laser central frequency inside
`process_pipeline` so the output is dimensionless fractional frequency.

In [ ]:
# ── Pipeline parameters ───────────────────────────────────────────────────────
downsample_kwargs = {
    "target_fs": 0.2,  # Hz — target sampling rate
    "kaiser_window": 31.0,  # Kaiser β for anti-aliasing
}
filter_kwargs = {
    "highpass_cutoff": 8e-5,  # Hz
    "lowpass_cutoff": 0.8 * downsample_kwargs["target_fs"],  # Hz
    "order": 2,
}
trim_kwargs = {
    "fraction": 0.02,  # fraction trimmed from each end after downsampling
}
truncate_kwargs = {
    "days": 15.0,  # split dataset into 15-day segments
}
window_kwargs = {
    "window": "tukey",
    "alpha": 0.0,
}
# ─────────────────────────────────────────────────────────────────────────────

processed_segments = mp.process_pipeline(
    data_gapped,
    downsample_kwargs=downsample_kwargs,
    filter_kwargs=filter_kwargs,
    trim_kwargs=trim_kwargs,
    truncate_kwargs=truncate_kwargs,
    window_kwargs=window_kwargs,
)

sp_0 = processed_segments["segment0"]
print(f"Processed segments: {list(processed_segments.keys())}")
print(f"segment0: N={sp_0.N}, fs={sp_0.fs} Hz, T={sp_0.T / 86400:.2f} days")

## 5. Compute Extended Mask

`gaps.compute_extended_mask` mirrors the pipeline's downsampling and trimming,
filters `1 − mask` through the same Butterworth, and flags samples where the
absolute leakage exceeds `contamination_threshold` as excluded.

Both outputs are kept for inspection before applying any further steps.

> **Note:** `fs_raw=data["fs"]` must be the raw (pre-downsample) sampling rate.
> `smoothed_mask` spans the full dataset so this cannot be inferred from the segment alone.

In [ ]:
extended_mask, gap_contamination = gaps.compute_extended_mask(
    smoothed_mask,
    sp_0,
    filter_kwargs,
    downsample_kwargs,
    trim_kwargs,
    fs_raw=data["fs"],
    contamination_threshold=5e-4,
    min_clean_hours=12.0,
)

ext_gap = 1 - extended_mask.mean()
print(f"Extended gap fraction : {ext_gap:.4%}")
print(
    f"Mask length: {len(extended_mask):,}  sp_0.N: {sp_0.N:,}  match: {len(extended_mask) == sp_0.N}"
)

In [ ]:
# ── Recover downsampled+trimmed original mask for plotting ────────────────────
from fractions import Fraction
from scipy import signal as scipy_signal

fs_raw = data["fs"]
target_fs = downsample_kwargs["target_fs"]
trim_frac = trim_kwargs["fraction"]
ratio = Fraction(target_fs / fs_raw).limit_denominator(10000)
ds = scipy_signal.resample_poly(
    smoothed_mask.astype(float), ratio.numerator, ratio.denominator
)
trim_n = int(round(len(ds) * trim_frac / 2))
smoothed_mask_ds = (ds[trim_n:-trim_n] if trim_n > 0 else ds)[: sp_0.N]

contamination_threshold = 1e-3
t_days = (sp_0.t - sp_0.t[0]) / 86400

# Zoom to ±3 days around the first gap
_diff = np.diff(np.concatenate([[1], extended_mask.astype(int), [1]]))
_gap_start = np.where(_diff == -1)[0]
_gap_end = np.where(_diff == 1)[0]

if len(_gap_start):
    _gc = (_gap_start[0] + _gap_end[0]) // 2
    _zoom = int(3 * 86400 * sp_0.fs)
else:
    _gc, _zoom = sp_0.N // 2, sp_0.N // 4
_lo, _hi = max(0, _gc - _zoom), min(sp_0.N, _gc + _zoom)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# ── Top panel: linear scale ──────────────────────────────────────────────────
ax1.plot(
    t_days[_lo:_hi], smoothed_mask_ds[_lo:_hi], label="Downsampled mask", alpha=0.7
)
ax1.plot(
    t_days[_lo:_hi],
    gap_contamination[_lo:_hi],
    label="Filter leakage [filtered(1−mask)]",
    alpha=0.9,
)
ax1.axhline(
    contamination_threshold,
    color="k",
    ls="--",
    label=f"±threshold = {contamination_threshold}",
)
ax1.axhline(-contamination_threshold, color="k", ls="--")
ax1.set_ylabel("Amplitude")
ax1.set_title("Butterworth filter leakage near first gap")
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# ── Bottom panel: log10 of absolute leakage vs threshold ─────────────────────
abs_contamination = np.abs(gap_contamination[_lo:_hi])
# Replace exact zeros with NaN so log10 doesn't produce -inf
abs_contamination[abs_contamination == 0] = np.nan

ax2.plot(
    t_days[_lo:_hi], np.log10(abs_contamination), label="|Filter leakage|", alpha=0.9
)
ax2.axhline(
    np.log10(contamination_threshold),
    color="k",
    ls="--",
    label=f"threshold = {contamination_threshold}",
)
ax2.set_xlabel("Time [days]")
ax2.set_ylabel("log$_{10}$(|Amplitude|)")
ax2.set_title("Absolute filter leakage (log scale)")
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Taper the Mask

`gaps.taper_mask` applies two independent half-cosine tapers:
- **Gap-edge taper** — smooth ramp at each `0→1` and `1→0` transition.
- **Dataset-endpoint taper** — ramp at the very start and end of the array.

In [ ]:
gap_mask = gaps.taper_mask(
    extended_mask,
    sp_0,
    taper_hours=7.0,
    edge_taper_hours=3.0,
)

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(
    t_days[_lo:_hi], extended_mask[_lo:_hi], label="Extended binary mask", alpha=0.8
)
ax.plot(t_days[_lo:_hi], gap_mask[_lo:_hi], label="Final tapered mask", alpha=0.9)
ax.set_xlabel("Time [days]")
ax.set_ylabel("Mask value")
ax.set_title("Final window function (gap taper=12 h, edge taper=12 h)")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Apply Final Mask

`gaps.apply_mask_to_processor` returns a **new** `SignalProcessor` with the
mask applied.  `sp_0` is never modified, so both the masked and unmasked
versions are available for comparison.

In [ ]:
sp_0_masked = gaps.apply_mask_to_processor(sp_0, gap_mask)
sp_0_masked_aet = sp_0_masked.to_aet()

print(f"Masked channels: {sp_0_masked.channels}")
print(f"AET channels:    {sp_0_masked_aet.channels}")

fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
for i, ch in enumerate(["X", "Y", "Z"]):
    axes[i].plot(
        t_days,
        sp_0_masked.data[ch],
        linewidth=0.4,
        color=f"C{i}",
        label=f"TDI-{ch} (masked)",
    )
    axes[i].set_ylabel("Frac. freq.", fontsize=12)
    axes[i].legend(loc="upper right", fontsize=11)
    axes[i].grid(True, alpha=0.3)
axes[2].set_xlabel("Time [days]", fontsize=12)
plt.tight_layout()
plt.show()

## 8. Compute FFT and Periodogram

The one-sided periodogram estimate of the noise PSD is

$$\hat{S}(f_k) = \frac{2\,\Delta t}{N} \left|\tilde{n}(f_k)\right|^2$$

where $\tilde{n}$ is the FFT of the masked time series, $\Delta t$ the sampling
interval and $N$ the segment length.

In [ ]:
CENTRAL_FREQ = data["metadata"]["laser_frequency"]

freq, fft_xyz = sp_0_masked.fft()
_, fft_aet = sp_0_masked_aet.fft()

psd_norm = 2 * sp_0_masked.dt / sp_0_masked.N
psd_xyz = {ch: np.abs(fft_xyz[ch]) ** 2 * psd_norm for ch in ["X", "Y", "Z"]}
psd_aet = {ch: np.abs(fft_aet[ch]) ** 2 * psd_norm for ch in ["A", "E", "T"]}

# Mojito L1 noise model — frequency axis derived from the noise covariance shape
noise_cov_xyz = data["noise_estimates"]["xyz"]
noise_cov_aet = data["noise_estimates"]["aet"]
noise_freqs = np.logspace(-5, 0, noise_cov_xyz[0].shape[0])

l1_xyz = {
    ch: noise_cov_xyz[0][:, i, i] / CENTRAL_FREQ**2
    for i, ch in enumerate(["X", "Y", "Z"])
}
l1_aet = {
    ch: noise_cov_aet[0][:, i, i] / CENTRAL_FREQ**2
    for i, ch in enumerate(["A", "E", "T"])
}

## 9. Periodogram vs Mojito L1 Estimate

In [ ]:
nyquist = sp_0_masked.fs / 2
xyz_colors = ["C0", "C1", "C2"]
aet_colors = ["#e377c2", "#9467bd", "#8c564b"]

fig, axes = plt.subplots(3, 2, figsize=(16, 10), sharex=False, sharey=True)

for i, ch in enumerate(["X", "Y", "Z"]):
    ax = axes[i, 0]
    ax.loglog(
        freq[1:], psd_xyz[ch][1:], linewidth=1.0, color=xyz_colors[i], label=f"TDI-{ch}"
    )
    ax.loglog(
        noise_freqs, l1_xyz[ch], ls="--", color="red", linewidth=2.0, label="Mojito L1"
    )
    ax.axvspan(1e-4, 1e-1, alpha=0.15, color="green", label="Science band")
    ax.set_xlim(1e-5, nyquist)
    ax.set_ylim(1e-53, 1e-35)
    ax.set_ylabel("PSD [1/Hz]", fontsize=20)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(loc="upper left", fontsize=14, framealpha=0.95)
    ax.tick_params(axis="both", which="major", labelsize=16)

for i, ch in enumerate(["A", "E", "T"]):
    ax = axes[i, 1]
    ax.loglog(
        freq[1:], psd_aet[ch][1:], linewidth=1.0, color=aet_colors[i], label=f"TDI-{ch}"
    )
    ax.loglog(noise_freqs, l1_aet[ch], ls="--", color="red", linewidth=2.0)
    ax.axvspan(1e-4, 1e-1, alpha=0.15, color="green")
    ax.set_xlim(1e-5, nyquist)
    ax.set_ylim(1e-53, 1e-35)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(loc="upper left", fontsize=14, framealpha=0.95)
    ax.tick_params(axis="both", which="major", labelsize=16)

axes[2, 0].set_xlabel("Frequency [Hz]", fontsize=20)
axes[2, 1].set_xlabel("Frequency [Hz]", fontsize=20)

fig.suptitle(
    f"Gapped Mojito Periodogram vs L1 Estimate (segment0)\n"
    f"HP={filter_kwargs['highpass_cutoff']:.0e} Hz, "
    f"LP={filter_kwargs['lowpass_cutoff']} Hz, "
    f"trim={trim_kwargs['fraction']:.1%}, "
    f"{truncate_kwargs['days']} days/segment, "
    f"fs={sp_0_masked.fs} Hz",
    fontsize=18,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Data retention analysis ───────────────────────────────────────────────────
raw_gap_frac = 1 - smoothed_mask.mean()
extended_gap_frac = 1 - extended_mask.mean()
final_mask_frac = 1 - gap_mask.mean()

# Segment duration
seg_days = sp_0.T / 86400

print("=" * 60)
print("Data loss budget (segment0)")
print("=" * 60)
print(f"{'Stage':<35} {'Gap %':>8}  {'Extra %':>8}")
print("-" * 60)
print(f"{'1. Raw gap mask (smoothed_mask)':<35} {raw_gap_frac:>8.2%}  {'—':>8}")
print(
    f"{'2. Extended mask (filter leakage)':<35} {extended_gap_frac:>8.2%}  {extended_gap_frac - raw_gap_frac:>+8.2%}"
)
print(
    f"{'3. Final tapered mask (gap_mask)':<35} {final_mask_frac:>8.2%}  {final_mask_frac - extended_gap_frac:>+8.2%}"
)
print("-" * 60)
print(
    f"{'Total extra data lost vs raw gaps:':<35} {final_mask_frac - raw_gap_frac:>+8.2%}"
)
print()

# Count individual gap regions in extended mask
_d = np.diff(np.concatenate([[1], extended_mask.astype(int), [1]]))
n_gaps = len(np.where(_d == -1)[0])
gap_lengths_hr = []
_starts = np.where(_d == -1)[0]
_ends = np.where(_d == 1)[0]
for s, e in zip(_starts, _ends):
    gap_lengths_hr.append((e - s) / sp_0.fs / 3600)
gap_lengths_hr = np.array(gap_lengths_hr)

print(f"Number of gaps in segment0: {n_gaps}")
if n_gaps:
    print(
        f"Gap durations: min={gap_lengths_hr.min():.1f} h, "
        f"median={np.median(gap_lengths_hr):.1f} h, "
        f"max={gap_lengths_hr.max():.1f} h"
    )
print()

# Current parameters summary
print("Current parameters:")
print(f"  contamination_threshold = 5e-4")
print(f"  min_clean_hours         = 12.0 h")
print(f"  taper_hours             = 7.0 h")
print(f"  edge_taper_hours        = 1.0 h")
print(f"  lisagap lobe_lengths_hr = 3.0 h")
print(f"  highpass_cutoff         = {filter_kwargs['highpass_cutoff']:.0e} Hz")

## 10. Windowed Noise Covariance - Leading Diagonal

From Burke et al. (arXiv:2502.17426, Algorithm 1, Eq. 59), the Fourier-domain
noise covariance for windowed data is

$$\left(\tilde{W}\,\Sigma\,\tilde{W}\right)_{ij} = \frac{\Delta f}{2}\sum_{k=0}^{N-1} S_n^k\,\tilde{w}_{\overline{i-k}}\,\tilde{w}^*_{\overline{j-k}}$$

where $\tilde{w} = \Delta t\cdot\mathrm{FFT}(w)$ and $S_n$ is the one-sided PSD.

For the **main diagonal** ($i = j$, i.e. $p = 0$) this reduces to a single
circular convolution of $S_n$ with $|\tilde{w}|^2$:

$$\tilde{\Sigma}_{ii} = \frac{\Delta f}{2}\,\mathrm{IFFT}\!\bigl(\mathrm{FFT}(u)\cdot\mathrm{FFT}(v)\bigr)_i$$

The expected one-sided periodogram is then $E[\hat{S}_i] = 2\,\Delta f\,\tilde{\Sigma}_{ii}$.

In [ ]:
from scipy.interpolate import interp1d


def windowed_psd_diagonal(Sn_onesided, w, dt):
    """Expected one-sided periodogram for windowed data (main diagonal only).

    Implements the p=0 diagonal of Algorithm 1 from Burke et al.
    (arXiv:2502.17426, Eq. 59).

    Parameters
    ----------
    Sn_onesided : array, shape (N//2+1,)
        One-sided PSD on the uniform FFT grid f_k = k * df.
    w : array, shape (N,)
        Time-domain window (e.g. gap_mask).
    dt : float
        Sampling interval in seconds.

    Returns
    -------
    array, shape (N//2+1,)
        Expected one-sided periodogram including window effects.
    """
    N = len(w)
    df = 1.0 / (N * dt)

    # Physical DFT of the window: w_tilde = dt * FFT(w)
    w_tilde = dt * np.fft.fft(w)

    # Two-sided PSD array (mirror one-sided onto negative frequencies)
    u = np.zeros(N)
    u[: N // 2 + 1] = Sn_onesided
    u[N // 2 + 1 :] = Sn_onesided[-2:0:-1]

    # Diagonal p=0: v_i = |w_tilde_i|^2
    v = np.abs(w_tilde) ** 2

    # Convolution via FFT -> covariance diagonal
    D = (df / 2.0) * np.fft.ifft(np.fft.fft(u) * np.fft.fft(v)).real

    # Covariance diagonal -> expected one-sided periodogram:
    #   E[S_hat_i] = (2 * dt / N) * E[|FFT(x)|^2] = 2 * df * Sigma_ii
    return 2.0 * df * D[: N // 2 + 1]


# ── Interpolate L1 PSD onto the uniform FFT frequency grid ───────────────────
N_seg = sp_0_masked.N
dt_seg = sp_0_masked.dt
df_seg = 1.0 / (N_seg * dt_seg)
fft_freqs = np.arange(N_seg // 2 + 1) * df_seg


def _interp_psd_loglog(model_freqs, model_psd, target_freqs):
    """Log-log interpolation of a PSD onto a new frequency grid."""
    log_interp = interp1d(
        np.log10(model_freqs),
        np.log10(model_psd),
        kind="linear",
        fill_value="extrapolate",
    )
    safe_freqs = np.maximum(target_freqs, model_freqs[0])
    out = 10.0 ** log_interp(np.log10(safe_freqs))
    out[0] = out[1]  # DC bin: use lowest positive-frequency value
    return out


Sn_fft_xyz = {
    ch: _interp_psd_loglog(noise_freqs, l1_xyz[ch], fft_freqs) for ch in ["X", "Y", "Z"]
}
Sn_fft_aet = {
    ch: _interp_psd_loglog(noise_freqs, l1_aet[ch], fft_freqs) for ch in ["A", "E", "T"]
}

# ── Compute windowed PSD diagonal ────────────────────────────────────────────
windowed_xyz = {
    ch: windowed_psd_diagonal(Sn_fft_xyz[ch], gap_mask, dt_seg)
    for ch in ["X", "Y", "Z"]
}
windowed_aet = {
    ch: windowed_psd_diagonal(Sn_fft_aet[ch], gap_mask, dt_seg)
    for ch in ["A", "E", "T"]
}

print(
    f"Windowed PSD diagonal computed: N={N_seg}, "
    f"df={df_seg:.2e} Hz, window=gap_mask"
)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 10), sharex=False, sharey=True)

for i, ch in enumerate(["X", "Y", "Z"]):
    ax = axes[i, 0]
    ax.loglog(
        freq[1:], psd_xyz[ch][1:], linewidth=1.0, color=xyz_colors[i], label=f"TDI-{ch}"
    )
    ax.loglog(
        noise_freqs,
        l1_xyz[ch],
        ls="--",
        color="red",
        linewidth=2.0,
        label="L1 (no window)",
    )
    ax.loglog(
        fft_freqs[1:],
        windowed_xyz[ch][1:],
        ls="--",
        color="blue",
        linewidth=2.0,
        label="L1 + window",
    )
    ax.axvspan(1e-4, 1e-1, alpha=0.15, color="green", label="Science band")
    ax.set_xlim(1e-5, nyquist)
    ax.set_ylim(1e-53, 1e-35)
    ax.set_ylabel("PSD [1/Hz]", fontsize=20)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(loc="upper left", fontsize=12, framealpha=0.95)
    ax.tick_params(axis="both", which="major", labelsize=16)

for i, ch in enumerate(["A", "E", "T"]):
    ax = axes[i, 1]
    ax.loglog(
        freq[1:], psd_aet[ch][1:], linewidth=1.0, color=aet_colors[i], label=f"TDI-{ch}"
    )
    ax.loglog(
        noise_freqs,
        l1_aet[ch],
        ls="--",
        color="red",
        linewidth=2.0,
        label="L1 (no window)",
    )
    ax.loglog(
        fft_freqs[1:],
        windowed_aet[ch][1:],
        ls="--",
        color="blue",
        linewidth=2.0,
        label="L1 + window",
    )
    ax.axvspan(1e-4, 1e-1, alpha=0.15, color="green")
    ax.set_xlim(1e-5, nyquist)
    ax.set_ylim(1e-53, 1e-35)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(loc="upper left", fontsize=12, framealpha=0.95)
    ax.tick_params(axis="both", which="major", labelsize=16)

axes[2, 0].set_xlabel("Frequency [Hz]", fontsize=20)
axes[2, 1].set_xlabel("Frequency [Hz]", fontsize=20)

fig.suptitle(
    "Periodogram vs L1 Estimate vs Windowed L1 (segment0)\n"
    r"$E[\hat{S}_i] = 2\Delta f\,\tilde{\Sigma}_{ii}$ "
    f"(Algorithm 1, diagonal only)",
    fontsize=18,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Diagnostic: why does windowed ≈ unwindowed? ──────────────────────────────
w = gap_mask
N = len(w)

# Window statistics
live_frac = np.sum(w > 0) / N
w2_frac = np.sum(w**2) / N  # ≈ live fraction for binary-ish mask

# Power distribution of |w_tilde|^2
w_tilde = sp_0_masked.dt * np.fft.fft(w)
v = np.abs(w_tilde) ** 2
dc_power_frac = v[0] / v.sum()

print("=== Window diagnostics ===")
print(f"  N              = {N:,}")
print(f"  Live fraction  = {live_frac:.4f}  (fraction of w > 0)")
print(f"  sum(w^2)/N     = {w2_frac:.4f}  (effective duty cycle)")
print(f"  DC power / total power in |w̃|² = {dc_power_frac:.6f}")
print(f"  Leakage power  = {1 - dc_power_frac:.6f}  spread over {N-1:,} bins")
print(f"  Avg leakage per bin / DC = {(1 - dc_power_frac) / (N - 1):.2e}")
print()

# Ratio windowed / unwindowed for each channel
print("=== Windowed / unwindowed PSD ratio (in science band 1e-4 to 0.1 Hz) ===")
science = (fft_freqs > 1e-4) & (fft_freqs < 0.1)
for ch in ["X", "Y", "Z", "A", "E", "T"]:
    wdict = windowed_xyz if ch in "XYZ" else windowed_aet
    sdict = Sn_fft_xyz if ch in "XYZ" else Sn_fft_aet
    ratio_vals = wdict[ch][science] / sdict[ch][science]
    print(
        f"  {ch}: mean={ratio_vals.mean():.6f}, "
        f"min={ratio_vals.min():.6f}, max={ratio_vals.max():.6f}"
    )

print(f"\n  Expected ratio for flat PSD: sum(w²)/N = {w2_frac:.6f}")
print(f"  On log-log scale: this is a shift of {np.log10(w2_frac):.4f} decades")
print(
    f"  Plot spans {53 - 35} = 18 decades → shift is "
    f"{abs(np.log10(w2_frac)) / 18 * 100:.2f}% of the y-axis range"
)

In [ ]:
# ── Ratio plot: windowed / unwindowed (reveals the small structure) ───────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for i, ch in enumerate(["X", "Y", "Z"]):
    ratio_ch = windowed_xyz[ch] / Sn_fft_xyz[ch]
    axes[0].semilogx(
        fft_freqs[1:],
        ratio_ch[1:].real,
        linewidth=0.6,
        color=xyz_colors[i],
        label=f"TDI-{ch}",
    )

axes[0].axhline(
    w2_frac, color="k", ls="--", lw=1.5, label=rf"$\sum w^2 / N = {w2_frac:.4f}$"
)
axes[0].axvspan(1e-4, 1e-1, alpha=0.1, color="green")
axes[0].set_xlabel("Frequency [Hz]", fontsize=14)
axes[0].set_ylabel(r"$E[\hat{S}_i^{\mathrm{win}}]\;/\;S_n^i$", fontsize=16)
axes[0].set_title("XYZ channels", fontsize=14)
axes[0].set_xlim(1e-5, nyquist)
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(labelsize=12)

for i, ch in enumerate(["A", "E", "T"]):
    ratio_ch = windowed_aet[ch] / Sn_fft_aet[ch]
    axes[1].semilogx(
        fft_freqs[1:],
        ratio_ch[1:].real,
        linewidth=0.6,
        color=aet_colors[i],
        label=f"TDI-{ch}",
    )

axes[1].axhline(
    w2_frac, color="k", ls="--", lw=1.5, label=rf"$\sum w^2 / N = {w2_frac:.4f}$"
)
axes[1].axvspan(1e-4, 1e-1, alpha=0.1, color="green")
axes[1].set_xlabel("Frequency [Hz]", fontsize=14)
axes[1].set_title("AET channels", fontsize=14)
axes[1].set_xlim(1e-5, nyquist)
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)
axes[1].tick_params(labelsize=12)

fig.suptitle(
    r"Ratio: windowed PSD diagonal / bare PSD"
    "\n"
    r"Flat $\rightarrow \sum w_j^2/N$;  "
    "deviations = spectral leakage redistributing power across bins",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

## 11. Off-Diagonal Analysis: Can We Neglect Cross-Frequency Correlations?

The $p$-th diagonal of the windowed covariance matrix is (Algorithm 1, Eq. 59):

$$\tilde{\Sigma}_{i,i+p} = \frac{\Delta f}{2}\,\mathrm{IFFT}\!\bigl(\mathrm{FFT}(u)\cdot\mathrm{FFT}(v_p)\bigr)_i, \qquad v_p[m] = \tilde{w}_m\,\tilde{w}^*_{\overline{m+p}}$$

We compute $|\tilde{\Sigma}_{i,i+p}| / \tilde{\Sigma}_{i,i}$ for several values of $p$ to determine whether the covariance is effectively diagonal for likelihood evaluation.

In [ ]:
def windowed_cov_diagonal_p(Sn_onesided, w, dt, p):
    """Compute the p-th diagonal of the windowed noise covariance matrix.

    Parameters
    ----------
    Sn_onesided : array, shape (N//2+1,)
        One-sided PSD on the uniform FFT grid.
    w : array, shape (N,)
        Time-domain window.
    dt : float
        Sampling interval in seconds.
    p : int
        Diagonal offset (p=0 is the main diagonal).

    Returns
    -------
    array, shape (N//2+1,)
        The p-th diagonal: Sigma_{i, i+p} for i = 0, ..., N//2.
    """
    N = len(w)
    df = 1.0 / (N * dt)

    w_tilde = dt * np.fft.fft(w)

    # Two-sided PSD
    u = np.zeros(N)
    u[: N // 2 + 1] = Sn_onesided
    u[N // 2 + 1 :] = Sn_onesided[-2:0:-1]

    # v_p[m] = w_tilde[m] * conj(w_tilde[(m+p) % N])
    v_p = w_tilde * np.conj(np.roll(w_tilde, -p))

    # Circular convolution
    D_p = (df / 2.0) * np.fft.ifft(np.fft.fft(u) * np.fft.fft(v_p))

    return D_p[: N // 2 + 1]


# ── Compute several off-diagonals for TDI-X (representative) ─────────────────
p_values = [0, 1, 2, 5, 10, 50, 100, 500, 1000]
Sn_X = Sn_fft_xyz["X"]

diags = {}
for p in p_values:
    diags[p] = windowed_cov_diagonal_p(Sn_X, gap_mask, dt_seg, p)

# Main diagonal (real, positive)
D0 = diags[0].real

print("=== Off-diagonal magnitude relative to main diagonal (TDI-X) ===")
print(
    f"{'p':>6s}  {'mean |D_p/D_0|':>16s}  {'max |D_p/D_0|':>16s}  "
    f"{'mean |D_p/D_0| (sci)':>22s}  {'max |D_p/D_0| (sci)':>22s}"
)
print("-" * 92)

science = (fft_freqs > 1e-4) & (fft_freqs < 0.1)
for p in p_values:
    ratio_all = np.abs(diags[p]) / D0
    ratio_sci = ratio_all[science]
    print(
        f"{p:6d}  {ratio_all[1:].mean():16.6e}  {ratio_all[1:].max():16.6e}  "
        f"{ratio_sci.mean():22.6e}  {ratio_sci.max():22.6e}"
    )

# ── Frobenius-norm contribution per diagonal ─────────────────────────────────
# ||Sigma||_F^2 = sum_p sum_i |Sigma_{i,i+p}|^2
# The fractional contribution of diagonal p is sum_i |D_p_i|^2 / ||Sigma||_F^2
# We compute the cumulative fraction to see how fast the off-diagonals decay.
print("\n=== Frobenius norm contribution (one-sided freqs, TDI-X) ===")
norm_D0_sq = np.sum(np.abs(diags[0][1:]) ** 2)
print(f"{'p':>6s}  {'||D_p||^2 / ||D_0||^2':>24s}  {'cumulative (p=0..p)':>22s}")
print("-" * 60)
cumulative = 0.0
for p in p_values:
    frac = np.sum(np.abs(diags[p][1:]) ** 2) / norm_D0_sq
    # For p>0 each off-diagonal appears twice (p and -p)
    cumulative += frac * (1 if p == 0 else 2)
    print(f"{p:6d}  {frac:24.6e}  {cumulative:22.6e}")

In [ ]:
# ── Sweep many p values and compute max|D_p/D_0| for decay profile ───────────
p_sweep = np.unique(
    np.concatenate(
        [
            np.arange(0, 20),
            np.arange(20, 100, 5),
            np.arange(100, 501, 50),
            np.arange(500, 2001, 500),
        ]
    )
).astype(int)

max_ratio = np.zeros(len(p_sweep))
mean_ratio_sci = np.zeros(len(p_sweep))
science = (fft_freqs > 1e-4) & (fft_freqs < 0.1)

for idx, p in enumerate(p_sweep):
    Dp = windowed_cov_diagonal_p(Sn_X, gap_mask, dt_seg, p)
    ratio_arr = np.abs(Dp[1:]) / D0[1:]
    max_ratio[idx] = ratio_arr.max()
    mean_ratio_sci[idx] = (np.abs(Dp[science]) / D0[science]).mean()

# ── Plot 1: Decay of off-diagonal magnitude vs p ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.semilogy(
    p_sweep, max_ratio, "o-", ms=3, label=r"max$_i\,|\Sigma_{i,i+p}|/\Sigma_{ii}$"
)
ax.semilogy(p_sweep, mean_ratio_sci, "s-", ms=3, label=r"mean$_i$ (science band)")
ax.axhline(0.01, color="red", ls="--", lw=1.5, label="1% threshold")
ax.axhline(0.001, color="orange", ls="--", lw=1.5, label="0.1% threshold")
ax.set_xlabel("Diagonal offset $p$", fontsize=14)
ax.set_ylabel(r"$|\Sigma_{i,i+p}| / \Sigma_{ii}$", fontsize=14)
ax.set_title("Off-diagonal decay (TDI-X)", fontsize=14)
ax.set_xlim(-5, p_sweep.max())
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.tick_params(labelsize=12)

# ── Plot 2: |D_p/D_0| vs frequency for selected p ──────────────────────────
ax = axes[1]
plot_ps = [1, 2, 5, 10, 50]
cmap = plt.cm.viridis(np.linspace(0, 0.9, len(plot_ps)))
for color, p in zip(cmap, plot_ps):
    Dp = diags[p]
    ax.semilogx(
        fft_freqs[1:],
        (np.abs(Dp[1:]) / D0[1:]).real,
        linewidth=0.8,
        color=color,
        label=f"$p={p}$",
    )
ax.axhline(0.01, color="red", ls="--", lw=1.5, alpha=0.6)
ax.axvspan(1e-4, 1e-1, alpha=0.1, color="green")
ax.set_xlabel("Frequency [Hz]", fontsize=14)
ax.set_ylabel(r"$|\Sigma_{i,i+p}| / \Sigma_{ii}$", fontsize=14)
ax.set_title(r"Off-diagonal ratio vs frequency (TDI-X)", fontsize=14)
ax.set_xlim(1e-5, nyquist)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.tick_params(labelsize=12)

fig.suptitle("Can the off-diagonals be neglected?", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Summary: at what p does the off-diagonal drop below thresholds? ──────────
for thresh_label, thresh in [("1%", 0.01), ("0.1%", 0.001)]:
    idx = np.where(max_ratio < thresh)[0]
    if len(idx):
        p_thresh = p_sweep[idx[0]]
        print(f"  max|D_p/D_0| < {thresh_label} for p >= {p_thresh}")
    else:
        print(f"  max|D_p/D_0| never drops below {thresh_label} in range tested")

In [ ]:
from scipy.linalg import eigvalsh, cholesky_banded

# ── Compute all diagonals up to max bandwidth ────────────────────────────────
max_bw = 200
Sn_X_real = Sn_fft_xyz["X"].real  # avoid ComplexWarning

print(f"Computing {max_bw+1} diagonals ... ", end="", flush=True)
all_diags = {}
for p in range(max_bw + 1):
    all_diags[p] = windowed_cov_diagonal_p(Sn_X_real, gap_mask, dt_seg, p)
print("done.")

# ── Pick a science-band subblock ─────────────────────────────────────────────
f_lo, f_hi = 1e-4, 0.02  # Hz — mid science band
i_lo = int(np.ceil(f_lo / df_seg))
i_hi = int(np.floor(f_hi / df_seg))
M = min(i_hi - i_lo + 1, 2000)
i_start = i_lo
print(
    f"Subblock: bins {i_start}–{i_start+M-1}, "
    f"freq {i_start*df_seg:.2e}–{(i_start+M-1)*df_seg:.2e} Hz, M={M}"
)

# ── Condition number vs bandwidth ────────────────────────────────────────────
bandwidths = [0, 1, 2, 5, 10, 20, 50, 100, 150, 200]

print(
    f"\n{'bw':>5s}  {'min(eig)':>12s}  {'max(eig)':>12s}  "
    f"{'cond':>12s}  {'PD?':>5s}"
)
print("-" * 55)

cond_numbers = []
for bw in bandwidths:
    Sigma = np.zeros((M, M), dtype=complex)
    for p in range(min(bw, max_bw) + 1):
        d = all_diags[p][i_start : i_start + M]
        idx = np.arange(M - p)
        if p == 0:
            Sigma[idx, idx] = d[:M].real
        else:
            Sigma[idx, idx + p] = d[: M - p]
            Sigma[idx + p, idx] = np.conj(d[: M - p])

    eigs = eigvalsh(Sigma)
    cond = eigs[-1] / eigs[0] if eigs[0] > 0 else np.inf
    is_pd = eigs[0] > 0
    cond_numbers.append(cond)
    print(
        f"{bw:5d}  {eigs[0]:12.4e}  {eigs[-1]:12.4e}  "
        f"{cond:12.4e}  {'Yes' if is_pd else 'NO':>5s}"
    )

# ── Also try banded Cholesky on the full one-sided grid (science band) ───────
print("\n=== Banded Cholesky on full science band ===")
sci_lo = int(np.ceil(1e-4 / df_seg))
sci_hi = int(np.floor(0.1 / df_seg))
M_full = sci_hi - sci_lo + 1
print(f"Full science band: {M_full:,} bins")

for bw in [10, 50, 100]:
    # Build banded storage: ab[bw-p, p:] for upper triangle
    ab = np.zeros((bw + 1, M_full), dtype=complex)
    for p in range(bw + 1):
        d = all_diags[p][sci_lo : sci_lo + M_full]
        if p == 0:
            ab[bw, :] = d.real
        else:
            ab[bw - p, p:] = d[: M_full - p]
    try:
        cholesky_banded(ab, lower=False, check_finite=False)
        print(f"  bw={bw:4d}: Cholesky succeeded (positive definite)")
    except np.linalg.LinAlgError:
        print(f"  bw={bw:4d}: Cholesky FAILED (not positive definite)")

In [ ]:
# ── Visualize: min eigenvalue & condition number vs bandwidth ─────────────────
bw_fine = list(range(0, 16)) + [20, 30, 50, 75, 100, 150, 200]
min_eigs = []
max_eigs = []

for bw in bw_fine:
    Sigma = np.zeros((M, M), dtype=complex)
    for p in range(min(bw, max_bw) + 1):
        d = all_diags[p][i_start : i_start + M]
        idx = np.arange(M - p)
        if p == 0:
            Sigma[idx, idx] = d[:M].real
        else:
            Sigma[idx, idx + p] = d[: M - p]
            Sigma[idx + p, idx] = np.conj(d[: M - p])
    eigs = eigvalsh(Sigma)
    min_eigs.append(eigs[0])
    max_eigs.append(eigs[-1])

min_eigs = np.array(min_eigs)
max_eigs = np.array(max_eigs)
bw_fine = np.array(bw_fine)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: min eigenvalue vs bandwidth
ax = axes[0]
ax.plot(bw_fine, min_eigs, "o-", ms=5, color="C0")
ax.axhline(0, color="red", ls="--", lw=1.5, label="PD boundary")
ax.fill_between(
    bw_fine,
    min_eigs,
    0,
    where=min_eigs < 0,
    alpha=0.2,
    color="red",
    label="NOT positive definite",
)
ax.fill_between(
    bw_fine,
    min_eigs,
    0,
    where=min_eigs >= 0,
    alpha=0.2,
    color="green",
    label="Positive definite",
)
ax.set_xlabel("Bandwidth (number of off-diagonals)", fontsize=14)
ax.set_ylabel(r"$\lambda_{\min}$", fontsize=16)
ax.set_title(
    "Minimum eigenvalue of banded $\\tilde{\\Sigma}$\n"
    f"(M={M} subblock, {i_start*df_seg:.1e}–{(i_start+M-1)*df_seg:.1e} Hz)",
    fontsize=13,
)
ax.legend(fontsize=11, loc="lower left")
ax.grid(True, alpha=0.3)
ax.tick_params(labelsize=12)

# Right: relative min eigenvalue (normalized by diagonal scale)
ax = axes[1]
diag_scale = all_diags[0][i_start : i_start + M].real.mean()
ax.semilogy(
    bw_fine,
    np.abs(min_eigs) / diag_scale,
    "o-",
    ms=5,
    color="C0",
    label=r"|$\lambda_{\min}$| / $\langle\Sigma_{ii}\rangle$",
)
ax.semilogy(
    bw_fine[min_eigs >= 0],
    min_eigs[min_eigs >= 0] / diag_scale,
    "o",
    ms=8,
    color="green",
    zorder=5,
    label="PD (positive)",
)
ax.semilogy(
    bw_fine[min_eigs < 0],
    np.abs(min_eigs[min_eigs < 0]) / diag_scale,
    "o",
    ms=8,
    color="red",
    zorder=5,
    label="Indefinite (negative)",
)
ax.set_xlabel("Bandwidth (number of off-diagonals)", fontsize=14)
ax.set_ylabel(r"|$\lambda_{\min}$| / $\langle\Sigma_{ii}\rangle$", fontsize=16)
ax.set_title("Relative magnitude of min eigenvalue", fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.tick_params(labelsize=12)

fig.suptitle(
    "Banded matrix conditioning: truncation breaks positive definiteness",
    fontsize=15,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

# ── Summary ──────────────────────────────────────────────────────────────────
pd_bws = bw_fine[min_eigs >= 0]
print(f"\nPD preserved for bandwidths: {list(pd_bws)}")
print(f"PD lost starting at bandwidth: {bw_fine[min_eigs < 0][0]}")
print(
    f"\nAt bw=200, |λ_min|/⟨Σ_ii⟩ = {abs(min_eigs[-1])/diag_scale:.2e} "
    f"(converging toward full PD matrix)"
)

## 12. Singular Covariance: Approximation Strategies for Likelihood Evaluation

The windowed covariance $\tilde{\Sigma} = \tilde{W}\Sigma\tilde{W}^\dagger$ is **singular** because
$w_j = 0$ in gaps ⟹ zero eigenvalues ⟹ $\tilde{\Sigma}^{-1}$ and $\det\tilde{\Sigma}$ undefined.

We compare four practical approximations:

| # | Method | Idea | Cost |
|---|--------|------|------|
| A | **Diagonal (Whittle)** | Ignore off-diagonals entirely: $\Sigma \approx \mathrm{diag}(\tilde{\Sigma}_{ii})$ | $\mathcal{O}(N)$ |
| B | **Pseudo-inverse** | SVD, discard zero eigenvalues, invert on non-null subspace | $\mathcal{O}(M^3)$ small block |
| C | **Time-domain restricted** | Only use observed samples; covariance is the Toeplitz submatrix of $C$ restricted to $w_j > 0$ | $\mathcal{O}(M_\mathrm{obs}^3)$ small block |
| D | **Woodbury correction** | Start from gap-free (circulant, fast), correct for the $N_\mathrm{gap}$ zeroed samples | $\mathcal{O}(N\log N + N_\mathrm{gap}^3)$ |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 12a. Eigenvalue spectrum of the full windowed covariance (subblock)
# ══════════════════════════════════════════════════════════════════════════════
from scipy.linalg import svd as scipy_svd, toeplitz

# Build the FULL windowed covariance on the 2000-bin subblock.
M_block = M  # 2000 from earlier
i0 = i_start

print(f"Building full {M_block}×{M_block} windowed covariance (science band) ...")
Sigma_full = np.zeros((M_block, M_block), dtype=complex)
for p in range(M_block):
    if p in all_diags:
        d = all_diags[p]
    else:
        d = windowed_cov_diagonal_p(Sn_X_real, gap_mask, dt_seg, p)
    idx_arr = np.arange(M_block - p)
    if p == 0:
        Sigma_full[idx_arr, idx_arr] = d[i0 : i0 + M_block].real
    else:
        Sigma_full[idx_arr, idx_arr + p] = d[i0 : i0 + M_block - p]
        Sigma_full[idx_arr + p, idx_arr] = np.conj(d[i0 : i0 + M_block - p])

eigs_full = eigvalsh(Sigma_full)

n_neg = np.sum(eigs_full < 0)
n_small = np.sum(eigs_full < 1e-4 * eigs_full.max())

print(f"  Eigenvalue range:  [{eigs_full[0]:.4e}, {eigs_full[-1]:.4e}]")
print(f"  Negative:          {n_neg}")
print(f"  λ < 1e-4·λ_max:   {n_small}")
print(f"  Effective rank:    {M_block - n_small} / {M_block}")

# ── Eigenvalue spectrum plot ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
sorted_abs = np.sort(np.abs(eigs_full))[::-1]
ax.semilogy(np.arange(M_block), sorted_abs, linewidth=0.8)
ax.axhline(
    1e-4 * eigs_full.max(), color="red", ls="--", label=rf"$10^{{-4}}\lambda_{{\max}}$"
)
ax.set_xlabel("Eigenvalue index (sorted)", fontsize=14)
ax.set_ylabel(r"$|\lambda_k|$", fontsize=16)
ax.set_title(f"Eigenvalue spectrum ({M_block}×{M_block} block)", fontsize=13)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

ax = axes[1]
sorted_pos = np.sort(eigs_full.real)[::-1]
sorted_pos[sorted_pos < 0] = 0
cumvar = np.cumsum(sorted_pos) / sorted_pos.sum()
ax.plot(np.arange(M_block), cumvar, linewidth=1.5)
ax.axhline(0.99, color="red", ls="--", label="99% of trace")
ax.axhline(0.999, color="orange", ls="--", label="99.9% of trace")
n_99 = np.searchsorted(cumvar, 0.99) + 1
n_999 = np.searchsorted(cumvar, 0.999) + 1
ax.axvline(n_99, color="red", ls=":", alpha=0.5)
ax.axvline(n_999, color="orange", ls=":", alpha=0.5)
ax.set_xlabel("Number of eigenvalues", fontsize=14)
ax.set_ylabel("Cumulative variance fraction", fontsize=14)
ax.set_title("How many eigenvalues carry the information?", fontsize=13)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

fig.suptitle("Windowed covariance eigenvalue structure", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\n  Eigenvalues for 99% of trace:  {n_99} / {M_block}")
print(f"  Eigenvalues for 99.9% of trace: {n_999} / {M_block}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 12b. Compare likelihood approximation strategies
# ══════════════════════════════════════════════════════════════════════════════
from scipy.linalg import eigh

eigvals, eigvecs = eigh(Sigma_full)

rng = np.random.default_rng(42)
pos = eigvals > 0
n_pos = pos.sum()
z = rng.standard_normal(n_pos) + 1j * rng.standard_normal(n_pos)
z /= np.sqrt(2)
d_sim = (eigvecs[:, pos] * np.sqrt(eigvals[pos])) @ z

print(f"Simulated noise: M={M_block}, rank={n_pos}, null={M_block - n_pos}")

# ── A. Diagonal (Whittle) ────────────────────────────────────────────────────
diag_S = np.diag(Sigma_full).real
good_A = diag_S > 1e-6 * diag_S.max()
logL_A = -0.5 * np.sum(
    np.abs(d_sim[good_A]) ** 2 / diag_S[good_A]
    + np.log(diag_S[good_A])
    + np.log(2 * np.pi)
)

# ── B. Pseudo-inverse (eigenvalue truncation) ────────────────────────────────
thresholds = [1e-2, 1e-4, 1e-6, 1e-8]
res_B = {}
for thr in thresholds:
    keep = eigvals > thr * eigvals.max()
    n_k = keep.sum()
    dp = eigvecs[:, keep].conj().T @ d_sim
    lL = -0.5 * np.sum(
        np.abs(dp) ** 2 / eigvals[keep] + np.log(eigvals[keep]) + np.log(2 * np.pi)
    )
    res_B[thr] = (lL, n_k)

# ── C. Time-domain restricted ────────────────────────────────────────────────
M_td = 500
Sn_2s = np.zeros(N_seg)
Sn_2s[: N_seg // 2 + 1] = Sn_X_real
Sn_2s[N_seg // 2 + 1 :] = Sn_X_real[-2:0:-1]
acf = np.fft.ifft(Sn_2s / (N_seg * dt_seg)).real

# Find a chunk that straddles a gap boundary (mix of observed + gapped)
_diff = np.diff(np.concatenate([[1], (gap_mask > 0.5).astype(int), [1]]))
_transitions = np.where(_diff != 0)[0]
if len(_transitions) >= 2:
    # Centre on the first 0->1 transition
    td_centre = _transitions[1]
    td_start = max(0, td_centre - M_td // 2)
else:
    td_start = 0
w_chunk = gap_mask[td_start : td_start + M_td]
obs_mask = w_chunk > 0.5
n_obs = obs_mask.sum()
print(f"\nTD restricted (M={M_td}): {n_obs} obs, {M_td - n_obs} gapped")

C_obs = toeplitz(acf[:M_td])[np.ix_(obs_mask, obs_mask)]
eig_C, vec_C = eigh(C_obs)
pos_C = eig_C > 1e-10 * eig_C.max()
x_all = sp_0.data["X"][td_start : td_start + M_td]
x_obs = x_all[obs_mask]
xp = vec_C[:, pos_C].T @ x_obs
logL_C = -0.5 * np.sum(xp**2 / eig_C[pos_C] + np.log(eig_C[pos_C]) + np.log(2 * np.pi))

# ── D. Woodbury / Schur complement ──────────────────────────────────────────
gap_idx_td = np.where(~obs_mask)[0]
obs_idx_td = np.where(obs_mask)[0]
n_gap_w = len(gap_idx_td)
print(f"Woodbury: {n_gap_w} gap samples ({n_gap_w/M_td:.1%} of block)")

C_full_td = toeplitz(acf[:M_td])
if n_gap_w > 0:
    C_oo = C_full_td[np.ix_(obs_idx_td, obs_idx_td)]
    C_og = C_full_td[np.ix_(obs_idx_td, gap_idx_td)]
    C_gg = C_full_td[np.ix_(gap_idx_td, gap_idx_td)]
    eig_gg, vec_gg = eigh(C_gg)
    C_gg_inv_Cgo = vec_gg @ (vec_gg.T @ C_og.T / eig_gg[:, None])
    C_schur = C_oo - C_og @ C_gg_inv_Cgo
    eig_D, vec_D = eigh(C_schur)
    pos_D = eig_D > 1e-10 * eig_D.max()
    xp_D = vec_D[:, pos_D].T @ x_obs
    logL_D = -0.5 * np.sum(
        xp_D**2 / eig_D[pos_D] + np.log(eig_D[pos_D]) + np.log(2 * np.pi)
    )
else:
    logL_D = logL_C
    pos_D = pos_C

# ── Summary ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 75)
print("LIKELIHOOD APPROXIMATION COMPARISON")
print("=" * 75)
print(f"\n{'Method':<45s} {'log L':>14s}  {'# modes':>8s}")
print("-" * 72)
print(f"{'A. Diagonal (Whittle)':<45s} {logL_A.real:14.2f}  {good_A.sum():8d}")
for thr in thresholds:
    lL, nk = res_B[thr]
    print(
        f"{'B. Pseudo-inv (lam/lam_max > ' + f'{thr:.0e})':<45s} {lL.real:14.2f}  {nk:8d}"
    )
print(f"{'C. TD restricted (obs samples only)':<45s} {logL_C:14.2f}  {pos_C.sum():8d}")
print(f"{'D. TD + Schur complement (Woodbury)':<45s} {logL_D:14.2f}  {pos_D.sum():8d}")
print(f"\n  A,B: Fourier domain, M={M_block} block")
print(f"  C,D: Time domain, M={M_td} block ({n_obs} obs, {n_gap_w} gapped)")
print(f"  C = naive restricted; D = conditioned on full covariance via Schur.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 12c. Fair comparison: all methods on the SAME simulated Fourier-domain data
# ══════════════════════════════════════════════════════════════════════════════
# Use d_sim (draw from the full windowed covariance) for all methods.
# This ensures log L values are directly comparable.


# Helper: log-likelihood on a given eigenbasis
def _logL_eig(d, eigvals_k, eigvecs_k):
    dp = eigvecs_k.conj().T @ d
    return (
        -0.5
        * np.sum(
            np.abs(dp) ** 2 / eigvals_k + np.log(eigvals_k) + np.log(2 * np.pi)
        ).real
    )


# ── Reference: full pseudo-inverse at machine precision ──────────────────────
keep_ref = eigvals > 1e-12 * eigvals.max()
logL_ref = _logL_eig(d_sim, eigvals[keep_ref], eigvecs[:, keep_ref])
n_ref = keep_ref.sum()

# ── A. Diagonal ──────────────────────────────────────────────────────────────
# Already computed: logL_A

# ── B. Pseudo-inverse at various thresholds ──────────────────────────────────
# Already computed: res_B

# ── E. Diagonal + rescale by w² (corrected Whittle) ──────────────────────────
# Correct the diagonal for the window power loss: Sigma_ii ≈ w2_frac * S_n(f_i)
# This is what most people actually use.
diag_bare = Sn_X_real[i0 : i0 + M_block]  # bare PSD (unwindowed)
good_E = diag_bare > 1e-6 * diag_bare.max()
logL_E = -0.5 * np.sum(
    np.abs(d_sim[good_E]) ** 2 / diag_bare[good_E]
    + np.log(diag_bare[good_E])
    + np.log(2 * np.pi)
)

# ── Print comparison ─────────────────────────────────────────────────────────
print("=" * 80)
print("FAIR COMPARISON: all methods on same simulated data from windowed covariance")
print(f"d_sim ~ N(0, Sigma_full),  M={M_block},  rank={n_pos}")
print("=" * 80)
print(f"\n{'Method':<50s} {'log L':>12s} {'#modes':>7s} {'Delta logL':>12s}")
print("-" * 85)
print(
    f"{'Reference (pseudo-inv, thr=1e-12)':<50s} "
    f"{logL_ref:12.1f} {n_ref:7d} {'0.0':>12s}"
)
print(
    f"{'A. Diagonal (windowed Sigma_ii)':<50s} "
    f"{logL_A.real:12.1f} {good_A.sum():7d} {logL_A.real - logL_ref:12.1f}"
)
print(
    f"{'E. Diagonal (bare PSD, unwindowed)':<50s} "
    f"{logL_E.real:12.1f} {good_E.sum():7d} {logL_E.real - logL_ref:12.1f}"
)
for thr in thresholds:
    lL, nk = res_B[thr]
    label = f"B. Pseudo-inv (thr={thr:.0e})"
    print(f"{label:<50s} {lL.real:12.1f} {nk:7d} {lL.real - logL_ref:12.1f}")

# ── Bar chart ────────────────────────────────────────────────────────────────
labels = [
    "Ref\n(full)",
    "A. Diag\n(windowed)",
    "E. Diag\n(bare PSD)",
    "B. 1e-2",
    "B. 1e-4",
    "B. 1e-6",
    "B. 1e-8",
]
logLs = [logL_ref, logL_A.real, logL_E.real, *[res_B[t][0].real for t in thresholds]]
deltas = [l - logL_ref for l in logLs]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
colors_bar = ["black", "C3", "C4", "C0", "C0", "C0", "C0"]
ax.bar(labels, logLs, color=colors_bar, alpha=0.7, edgecolor="k")
ax.set_ylabel("log L", fontsize=14)
ax.set_title("Absolute log-likelihood", fontsize=13)
ax.tick_params(labelsize=11)
ax.grid(axis="y", alpha=0.3)

ax = axes[1]
ax.bar(labels[1:], deltas[1:], color=colors_bar[1:], alpha=0.7, edgecolor="k")
ax.axhline(0, color="black", lw=1)
ax.set_ylabel(r"$\Delta\log L$ (vs reference)", fontsize=14)
ax.set_title("Likelihood bias relative to full pseudo-inverse", fontsize=13)
ax.tick_params(labelsize=11)
ax.grid(axis="y", alpha=0.3)

fig.suptitle(
    "Which approximation introduces least bias?", fontsize=15, fontweight="bold"
)
plt.tight_layout()
plt.show()

# ── Quantify per-mode bias ───────────────────────────────────────────────────
print(f"\nPer-mode bias (Delta logL / # modes):")
print(f"  A. Diagonal (windowed): {(logL_A.real - logL_ref)/good_A.sum():.4f}")
print(f"  E. Diagonal (bare PSD): {(logL_E.real - logL_ref)/good_E.sum():.4f}")
for thr in thresholds:
    lL, nk = res_B[thr]
    print(f"  B. Pseudo-inv {thr:.0e}:    {(lL.real - logL_ref)/nk:.4f}")